<a href="https://colab.research.google.com/github/Leidi-Lopez/Actividad03---Casos-con-Listas/blob/main/ProgIIIG1_Act04_Leidi_Jhoana_Lopez_Soto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programación III - Grupo 101 | Actividad 04
**Estudiante:** Leidi Jhoana Lopez Soto  
**Problema:** `SEND + MORE = MONEY` (CSP)

### Ajustes realizados al código de clase:
* **Depuración de la clase:** Corregí `ConsistenciaGlobal` para que trabaje con los atributos de la instancia (`self`) y eliminé el `input()` que frenaba la ejecución.
* **Restricciones del problema:** Ajusté las cotas (`S > 7`, `O < 2`, `M < 2`, `S != 0`, `M != 0`), agregué `O < S`, validé `TodosDiferentes` y conecté los acarreos (`C1`, `C2`, `C3`) entre columnas.
* **Backtracking:** Agregué las validaciones de estado completo, consistencia parcial y comprobación final de la suma.
* **Heurística:** Implementé selección por dominio más pequeño (MRV) con desempate por grado.

In [6]:
import copy
import time
from collections import Counter


class CriptoArithmCSP:
    def __init__(self, palabras, valores, acarreos=None):
        self.palabras = palabras
        self.letras = sorted(list(set(c for c in palabras if c.isalpha())))
        self.acarreos = list(acarreos) if acarreos else []
        self.variables = self.letras + self.acarreos
        self.valores = set(valores)

        # Dominios: {0..9} para las letras y {0, 1} para los acarreos entre columnas
        self.VarDom = {k: self.valores.copy() for k in self.letras}
        for c in self.acarreos:
            self.VarDom[c] = {0, 1}

        self.Restricciones = []
        self.soluciones = []

    def copy(self):
        return copy.deepcopy(self)

    def setRestriccion(self, restriccion):
        self.Restricciones.append(restriccion)

    # --- Restricciones locales (podan dominios sin fijar valores a mano) ---

    def MenorQue_PorValor(self, variables, valor):
        AnyChange = False
        for var in variables:
            OldDom = self.VarDom[var].copy()
            self.VarDom[var].difference_update(set(range(valor, 10)))
            if OldDom != self.VarDom[var]:
                AnyChange = True
        return AnyChange

    def MayorQue_PorValor(self, variables, valor):
        AnyChange = False
        for var in variables:
            OldDom = self.VarDom[var].copy()
            self.VarDom[var].difference_update(set(range(valor + 1)))
            if OldDom != self.VarDom[var]:
                AnyChange = True
        return AnyChange

    def DiferenteDe_PorValor(self, variables, valor):
        AnyChange = False
        for var in variables:
            if valor in self.VarDom[var]:
                self.VarDom[var].remove(valor)
                AnyChange = True
        return AnyChange

    def MenorQue_Variables(self, variables):
        v1, v2 = variables[0], variables[1]
        if not self.VarDom[v1] or not self.VarDom[v2]:
            return False
        AnyChange = False

        max_v2 = max(self.VarDom[v2])
        nuevo_v1 = {x for x in self.VarDom[v1] if x < max_v2}
        if nuevo_v1 != self.VarDom[v1]:
            self.VarDom[v1] = nuevo_v1
            AnyChange = True

        if not self.VarDom[v1]:
            return AnyChange

        min_v1 = min(self.VarDom[v1])
        nuevo_v2 = {y for y in self.VarDom[v2] if y > min_v1}
        if nuevo_v2 != self.VarDom[v2]:
            self.VarDom[v2] = nuevo_v2
            AnyChange = True

        return AnyChange

    def TodosDiferentes(self, variables):
        AnyChange = False

        # 1. Si una letra ya tiene un único dígito, se lo quitamos a las demás
        asignados = {
            var: next(iter(self.VarDom[var]))
            for var in variables
            if len(self.VarDom[var]) == 1
        }
        for var_asig, val in asignados.items():
            for otra_var in variables:
                if otra_var != var_asig and val in self.VarDom[otra_var]:
                    self.VarDom[otra_var].remove(val)
                    AnyChange = True

        # 2. Pares desnudos: solo podar si exactamente 2 variables comparten los mismos 2 dígitos
        pares = [self.VarDom[v] for v in variables if len(self.VarDom[v]) == 2]
        conteo = Counter(tuple(sorted(p)) for p in pares)

        for par, cantidad in conteo.items():
            if cantidad == 2:
                conjunto_par = set(par)
                variables_par = [v for v in variables if self.VarDom[v] == conjunto_par]
                for otra_var in variables:
                    if otra_var not in variables_par:
                        antes = self.VarDom[otra_var].copy()
                        self.VarDom[otra_var].difference_update(conjunto_par)
                        if antes != self.VarDom[otra_var]:
                            AnyChange = True
            elif cantidad > 2:
                # Si más de 2 letras comparten solo 2 valores posibles, el estado no es válido
                self.VarDom[variables[0]].clear()
                return True

        return AnyChange

    def RestriccionColumna(self, variables, var_c_in, var_c_out, acarreo_es_resultado=False):
        # Evalúa: variables[0] + variables[1] + c_in == variables[2] + 10 * c_out
        v1, v2, v_res = variables[0], variables[1], variables[2]
        sop_1, sop_2, sop_res = set(), set(), set()
        sop_cin, sop_cout = set(), set()

        dom_cin = {0} if var_c_in is None else self.VarDom[var_c_in]
        dom_cout = {1} if acarreo_es_resultado else self.VarDom[var_c_out]

        for a in self.VarDom[v1]:
            for b in self.VarDom[v2]:
                if v1 != v2 and a == b:
                    continue
                for c_in in dom_cin:
                    suma = a + b + c_in
                    c_out = suma // 10
                    r = suma % 10
                    if c_out not in dom_cout or r not in self.VarDom[v_res]:
                        continue
                    if acarreo_es_resultado and c_out != b:
                        continue
                    if v_res != v1 and r == a:
                        continue
                    if v_res != v2 and r == b:
                        continue
                    sop_1.add(a)
                    sop_2.add(b)
                    sop_res.add(r)
                    sop_cin.add(c_in)
                    sop_cout.add(c_out)

        AnyChange = False
        actualizaciones = [(v1, sop_1), (v2, sop_2), (v_res, sop_res)]
        if var_c_in is not None:
            actualizaciones.append((var_c_in, sop_cin))
        if var_c_out is not None:
            actualizaciones.append((var_c_out, sop_cout))

        for var, soporte in actualizaciones:
            if self.VarDom[var] != soporte:
                self.VarDom[var] = soporte
                AnyChange = True
        return AnyChange

    # --- Propagación de restricciones hasta que ningún dominio cambie ---

    def ConsistenciaGlobal(self, mostrar_traza=False):
        iteracion = 0
        while True:
            iteracion += 1
            AnyChange = False
            for restriccion in self.Restricciones:
                nombre_metodo = restriccion[0]
                parametros = restriccion[1:]
                metodo = getattr(self, nombre_metodo)
                if metodo(*parametros):
                    AnyChange = True
                    if mostrar_traza:
                        print(f"[Iter {iteracion}] {nombre_metodo}{parametros} -> {self.VarDom}")
                if not self.esConsistente():
                    return False
            if not AnyChange:
                break
        return True

    # --- Heurística MRV (dominio más pequeño) + desempate por Grado ---

    def _gradoVariable(self, var):
        grado = 0
        for restriccion in self.Restricciones:
            vars_r = restriccion[1]
            if var in vars_r:
                grado += sum(1 for v in vars_r if v != var and len(self.VarDom[v]) > 1)
        return grado

    def SeleccionarVariable(self, usar_heuristica=True):
        no_asignadas = [var for var in self.variables if len(self.VarDom[var]) > 1]
        if not no_asignadas:
            return None
        if not usar_heuristica:
            return no_asignadas[0]
        return min(
            no_asignadas,
            key=lambda var: (len(self.VarDom[var]), -self._gradoVariable(var)),
        )

    # --- Validaciones y Backtracking ---

    def esConsistente(self):
        asignados = []
        for var in self.variables:
            if len(self.VarDom[var]) == 0:
                return False
        for var in self.letras:
            if len(self.VarDom[var]) == 1:
                asignados.append(next(iter(self.VarDom[var])))
        if len(asignados) != len(set(asignados)):
            return False
        if self.VarDom["S"] == {0} or self.VarDom["M"] == {0}:
            return False
        return True

    def esCSPcompleto(self):
        return all(len(self.VarDom[var]) == 1 for var in self.variables)

    def esSolucion(self):
        if not self.esCSPcompleto() or not self.esConsistente():
            return False
        LetraValor = {var: next(iter(self.VarDom[var])) for var in self.letras}
        if len(set(LetraValor.values())) != len(self.letras):
            return False
        if LetraValor["S"] == 0 or LetraValor["M"] == 0:
            return False

        SEND = LetraValor["S"] * 1000 + LetraValor["E"] * 100 + LetraValor["N"] * 10 + LetraValor["D"]
        MORE = LetraValor["M"] * 1000 + LetraValor["O"] * 100 + LetraValor["R"] * 10 + LetraValor["E"]
        MONEY = LetraValor["M"] * 10000 + LetraValor["O"] * 1000 + LetraValor["N"] * 100 + LetraValor["E"] * 10 + LetraValor["Y"]
        return (SEND + MORE) == MONEY

    def backtraking(self, usar_heuristica=True, metricas=None):
        if metricas is None:
            metricas = {"nodos": 0, "ramificacion": []}
        metricas["nodos"] += 1

        if self.esCSPcompleto():
            if self.esSolucion():
                sol = {v: next(iter(self.VarDom[v])) for v in self.letras}
                if sol not in self.soluciones:
                    self.soluciones.append(sol)
                return True
            return False

        variableSeleccionadas = self.SeleccionarVariable(usar_heuristica=usar_heuristica)
        if variableSeleccionadas is None:
            return False

        metricas["ramificacion"].append(
            (variableSeleccionadas, len(self.VarDom[variableSeleccionadas]))
        )

        for valor in sorted(self.VarDom[variableSeleccionadas]):
            NuevoEstado = self.copy()
            NuevoEstado.VarDom[variableSeleccionadas] = {valor}

            if NuevoEstado.ConsistenciaGlobal() and NuevoEstado.esConsistente():
                if NuevoEstado.esCSPcompleto():
                    if NuevoEstado.esSolucion():
                        sol = {v: next(iter(NuevoEstado.VarDom[v])) for v in NuevoEstado.letras}
                        if sol not in self.soluciones:
                            self.soluciones.append(sol)
                else:
                    NuevoEstado.backtraking(usar_heuristica=usar_heuristica, metricas=metricas)
                    for sol in NuevoEstado.soluciones:
                        if sol not in self.soluciones:
                            self.soluciones.append(sol)

        return len(self.soluciones) > 0

In [7]:
def crear_modelo_send_more(con_columnas=True):
    acarreos = ["C1", "C2", "C3"] if con_columnas else []
    SendMore = CriptoArithmCSP("SEND+MORE=MONEY", set(range(10)), acarreos=acarreos)

    # Restricciones unarias
    SendMore.setRestriccion(("DiferenteDe_PorValor", ["S", "M"], 0))  # No empiezan en 0
    SendMore.setRestriccion(("MenorQue_PorValor", ["M"], 2))          # M < 2 (acarreo máximo = 1)
    SendMore.setRestriccion(("MayorQue_PorValor", ["S"], 7))          # S > 7 (corrige S < 8 de clase)
    SendMore.setRestriccion(("MenorQue_PorValor", ["O"], 2))          # O < 2 (corrige O < 6 de clase)

    # Restricciones binarias
    SendMore.setRestriccion(("MenorQue_Variables", ["O", "S"]))       # O < S
    SendMore.setRestriccion(("TodosDiferentes", SendMore.letras))     # Todas las letras distintas

    # Ecuaciones por columna conectando acarreos C1, C2 y C3
    if con_columnas:
        SendMore.setRestriccion(("RestriccionColumna", ["D", "E", "Y"], None, "C1", False))
        SendMore.setRestriccion(("RestriccionColumna", ["N", "R", "E"], "C1", "C2", False))
        SendMore.setRestriccion(("RestriccionColumna", ["E", "O", "N"], "C2", "C3", False))
        SendMore.setRestriccion(("RestriccionColumna", ["S", "M", "O"], "C3", None, True))

    return SendMore


SendMore = crear_modelo_send_more(con_columnas=True)
SendMore.ConsistenciaGlobal(mostrar_traza=True)

print("\nDominios finales tras ConsistenciaGlobal:")
for var in SendMore.variables:
    print(f"  {var}: {SendMore.VarDom[var]}")

[Iter 1] DiferenteDe_PorValor(['S', 'M'], 0) -> {'D': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'E': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'M': {1, 2, 3, 4, 5, 6, 7, 8, 9}, 'N': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'O': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'R': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'S': {1, 2, 3, 4, 5, 6, 7, 8, 9}, 'Y': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'C1': {0, 1}, 'C2': {0, 1}, 'C3': {0, 1}}
[Iter 1] MenorQue_PorValor(['M'], 2) -> {'D': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'E': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'M': {1}, 'N': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'O': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'R': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'S': {1, 2, 3, 4, 5, 6, 7, 8, 9}, 'Y': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'C1': {0, 1}, 'C2': {0, 1}, 'C3': {0, 1}}
[Iter 1] MayorQue_PorValor(['S'], 7) -> {'D': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'E': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'M': {1}, 'N': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'O': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'R': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'S': {8, 9}, 'Y': {0, 1, 2, 3, 4, 

In [8]:
# 1. Resolución con propagación por columnas + heurística MRV y Grado
metricas_k = {"nodos": 0, "ramificacion": []}
exito = SendMore.backtraking(usar_heuristica=True, metricas=metricas_k)

if exito:
    sol = SendMore.soluciones[0]
    print("Solución única válida encontrada:")
    print(" ", sol)
    print(f"  Comprobación: {sol['S']}{sol['E']}{sol['N']}{sol['D']} + "
          f"{sol['M']}{sol['O']}{sol['R']}{sol['E']} = "
          f"{sol['M']}{sol['O']}{sol['N']}{sol['E']}{sol['Y']}")
    print(f"  Nodos explorados: {metricas_k['nodos']}")
    print(f"  Orden de selección (variable, tamaño_dominio): {metricas_k['ramificacion']}")

# 2. Comparativa de la heurística MRV + Grado vs orden fijo (sin columnas)
csp_mrv = crear_modelo_send_more(con_columnas=False)
csp_mrv.ConsistenciaGlobal()
m_mrv = {"nodos": 0, "ramificacion": []}
t0 = time.perf_counter()
csp_mrv.backtraking(usar_heuristica=True, metricas=m_mrv)
t_mrv = (time.perf_counter() - t0) * 1000

csp_sin = crear_modelo_send_more(con_columnas=False)
csp_sin.ConsistenciaGlobal()
m_sin = {"nodos": 0, "ramificacion": []}
t0 = time.perf_counter()
csp_sin.backtraking(usar_heuristica=False, metricas=m_sin)
t_sin = (time.perf_counter() - t0) * 1000

print("\nComparativa de eficiencia de la heurística (dominio más pequeño):")
print(f"  Con Heurística MRV + Grado: {m_mrv['nodos']} nodos explorados ({t_mrv:.2f} ms)")
print(f"  Sin Heurística (orden fijo): {m_sin['nodos']} nodos explorados ({t_sin:.2f} ms)")

Solución única válida encontrada:
  {'D': 7, 'E': 5, 'M': 1, 'N': 6, 'O': 0, 'R': 8, 'S': 9, 'Y': 2}
  Comprobación: 9567 + 1085 = 10652
  Nodos explorados: 2
  Orden de selección (variable, tamaño_dominio): [('C1', 2), ('D', 2)]

Comparativa de eficiencia de la heurística (dominio más pequeño):
  Con Heurística MRV + Grado: 2201 nodos explorados (1367.46 ms)
  Sin Heurística (orden fijo): 2403 nodos explorados (2358.89 ms)
